Group 15: Diego Shan, Yan Li, Efe Aras <br>

<h1>Dataset<h1>


In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from collections import Counter
import numpy as np

from sst2 import CLASSES, load_sst2, get_tokenizer, get_cv_splits

<h3>Load<h3>

In [2]:
train, validation, test = load_sst2()

print(CLASSES)
print("train:", len(train))
print("validation:", len(validation))
print("test:", len(test))
print(train[2])

0: negative, 1: positive
train: 67349
validation: 872
test: 1821
('that loves its characters and communicates something rather beautiful about human nature ', 1)


<h3>Labels per split<h3>

In [3]:
for name, pairs in [("train", train), ("validation", validation), ("test", test)]:
    counts = Counter(label for text, label in pairs)
    print(name)
    for label, count in sorted(counts.items()):
        print(f"  {label}: {count} ({100 * count / len(pairs):.1f}%)")

train
  0: 29780 (44.2%)
  1: 37569 (55.8%)
validation
  0: 428 (49.1%)
  1: 444 (50.9%)
test
  -1: 1821 (100.0%)


<h3>Sentence lengths (words)<h3>

In [4]:
for name, pairs in [("train", train), ("validation", validation)]:
    lengths = np.array([len(text.split()) for text, label in pairs])
    print(f"{name}: mean {lengths.mean():.1f}, median {np.median(lengths):.0f}, min {lengths.min()}, max {lengths.max()}")

train: mean 9.4, median 7, min 1, max 52
validation: mean 19.5, median 19, min 2, max 47


<h3>Duplicates and short phrases in train<h3>

In [5]:
train_texts = [text for text, label in train]
print("unique sentences:", len(set(train_texts)), "of", len(train_texts))
print("phrases of 1-3 words:", sum(len(text.split()) <= 3 for text in train_texts))

validation_texts = set(text for text, label in validation)
print("validation sentences also in train:", sum(text in validation_texts for text in set(train_texts)))

unique sentences: 66978 of 67349
phrases of 1-3 words: 18542
validation sentences also in train: 0


<h3>Tokenizer<h3>

In [6]:
tokenizer = get_tokenizer(train)

print(tokenizer.get_vocab_size())
print(tokenizer.encode(train[2][0]).tokens)

5000
['▁that', '▁loves', '▁its', '▁characters', '▁and', '▁commun', 'ic', 'ates', '▁something', '▁rather', '▁beautiful', '▁about', '▁human', '▁nature', '▁']


<h3>Cross-validation folds<h3>

In [7]:
cv_sets = get_cv_splits(train)

for fold, split in cv_sets[0].items():
    print(fold, len(split["training"]), len(split["validation"]))

0 53879 13470
1 53879 13470
2 53879 13470
3 53879 13470
4 53880 13469
